In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

In [ ]:
# ==============================================================
# 1️⃣ Hardware-Safe Config
# ==============================================================
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
NUM_LABELS = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ==============================================================
# 2️⃣ Load and Split Data
# ==============================================================
# Ensure the file 'goemotions_final_7.csv' is in the same folder
df = pd.read_csv("goemotions_final_7.csv")

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Emotion_Encoded'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Emotion_Encoded'])

In [ ]:
# ==============================================================
# 3️⃣ Dataset Class
# ==============================================================
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# ==============================================================
# 4️⃣ Dataloaders
# ==============================================================
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(EmotionDataset(train_df['TEXT'].values, train_df['Emotion_Encoded'].values, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_df['TEXT'].values, val_df['Emotion_Encoded'].values, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmotionDataset(test_df['TEXT'].values, test_df['Emotion_Encoded'].values, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

In [ ]:
# ==============================================================
# 5️⃣ Model & Optimizer (System Safe)
# ==============================================================
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)

# Use the PyTorch AdamW for best performance
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = nn.CrossEntropyLoss().to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# ==============================================================
# 6️⃣ Training Loop (With History Saving)
# ==============================================================
history = {'train_acc': [], 'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    model.train()
    losses, correct = [], 0

    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        _, preds = torch.max(outputs.logits, dim=1)

        correct += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Safety check
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    # Validation
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=mask)
            _, preds = torch.max(outputs.logits, dim=1)
            val_correct += torch.sum(preds == labels)

    train_acc = correct.double().item() / len(train_df)
    val_acc = val_correct.double().item() / len(val_df)

    history['train_acc'].append(train_acc)
    history['train_loss'].append(np.mean(losses))
    history['val_acc'].append(val_acc)

    print(f"Train Loss: {history['train_loss'][-1]:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Save the model and the training history
torch.save(model.state_dict(), 'emotion_model_bert.pth')
pd.DataFrame(history).to_csv('training_history.csv', index=False)
print("\n✅ Training complete. Saved model and history.")


Epoch 1/4


Training:   0%|          | 0/3446 [00:00<?, ?it/s]

Training: 100%|██████████| 3446/3446 [4:37:32<00:00,  4.83s/it]  


Train Loss: 1.1378 | Train Acc: 0.5683 | Val Acc: 0.5697

Epoch 2/4


Training: 100%|██████████| 3446/3446 [4:35:29<00:00,  4.80s/it]  


Train Loss: 0.9984 | Train Acc: 0.6245 | Val Acc: 0.5722

Epoch 3/4


Training: 100%|██████████| 3446/3446 [4:35:31<00:00,  4.80s/it]  


Train Loss: 0.8680 | Train Acc: 0.6749 | Val Acc: 0.5602

Epoch 4/4


Training: 100%|██████████| 3446/3446 [4:35:38<00:00,  4.80s/it]  


Train Loss: 0.7555 | Train Acc: 0.7169 | Val Acc: 0.5513

✅ Training complete. Saved model and history.
